# Notebook 11 — Índices ENSO globales (ONI y SOI) como forzante climático complementario

Descarga directamente del Climate Prediction Center de la NOAA los índices ONI (Oceanic Niño Index, basado en la temperatura superficial del Pacífico ecuatorial 3.4) y SOI (Southern Oscillation Index, basado en el gradiente de presión Tahití-Darwin), los cruza con las anomalías NDVI del manglar de la CGSM y reporta correlaciones por rezago temporal. A diferencia del forzamiento ERA5-Land del notebook `07_era5_clima.ipynb` ---que promedia precipitación y temperatura sobre el área de estudio---, los índices ENSO ofrecen una medida directa del estado de la oscilación del Pacífico que define la variabilidad climática regional sin requerir promediado espacial, en este sentido complementan el análisis previo.

**Insumos:**
- ONI: https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt
- SOI: https://www.cpc.ncep.noaa.gov/data/indices/soi
- `outputs/tables/serie_temporal_ndvi_definitiva.csv`

**Productos:**
- `outputs/tables/indices_enso_mensual.csv`
- `outputs/tables/correlacion_enso_ndvi.csv`
- `outputs/figures/enso_serie_2013_2025.png`
- `outputs/figures/enso_vs_ndvi_correlacion.png`

In [ ]:
import urllib.request
from io import StringIO
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
OUT_TAB = ROOT / 'outputs' / 'tables'
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

URL_ONI = 'https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt'
URL_SOI = 'https://www.cpc.ncep.noaa.gov/data/indices/soi'
print(f'Outputs: {OUT_TAB}, {OUT_FIG}')

## 1. Descarga y parseo del ONI

El ONI se distribuye como tabla de texto con columnas `SEAS YR TOTAL ANOM`, donde cada fila corresponde a un trimestre móvil centrado en un mes (DJF para diciembre--enero--febrero, etc.). Se asocia el valor de cada trimestre al mes central.

In [ ]:
urllib.request.urlretrieve(URL_ONI, OUT_TAB / 'oni_raw.txt')
print('Descargado ONI raw')

# Parsear: SEAS YR TOTAL ANOM
oni_raw = pd.read_csv(OUT_TAB / 'oni_raw.txt', sep=r'\s+')

# Mapeo trimestre central -> mes (DJF=Jan, JFM=Feb, FMA=Mar, ...)
seas_to_month = {
    'DJF': 1, 'JFM': 2, 'FMA': 3, 'MAM': 4, 'AMJ': 5, 'MJJ': 6,
    'JJA': 7, 'JAS': 8, 'ASO': 9, 'SON': 10, 'OND': 11, 'NDJ': 12,
}
oni_raw['month'] = oni_raw['SEAS'].map(seas_to_month)
oni_raw['date'] = pd.to_datetime(
    oni_raw['YR'].astype(str) + '-' + oni_raw['month'].astype(str) + '-15')
oni = oni_raw[['date', 'ANOM']].rename(columns={'ANOM': 'oni'}).sort_values('date').reset_index(drop=True)

# Filtrar al periodo del proyecto
oni = oni[(oni['date'] >= '2013-01-01') & (oni['date'] <= '2025-12-31')].copy()
print(f'ONI: {len(oni)} registros, rango {oni.oni.min():.2f} a {oni.oni.max():.2f}')
print(oni.tail())

## 2. Descarga y parseo del SOI

El SOI se distribuye como matriz con columnas `YEAR JAN FEB ... DEC` y dos bloques (anomalía estandarizada arriba, anomalía cruda abajo). Se usa la versión estandarizada del bloque superior.

In [ ]:
urllib.request.urlretrieve(URL_SOI, OUT_TAB / 'soi_raw.txt')
print('Descargado SOI raw')

# El archivo tiene dos bloques; usamos el primero (anomalía estandarizada)
with open(OUT_TAB / 'soi_raw.txt') as f:
    lines = f.readlines()

# Encontrar la línea de encabezado YEAR JAN FEB ... DEC del bloque estandarizado
header_idx = None
for i, line in enumerate(lines):
    if line.strip().startswith('YEAR') and 'JAN' in line and 'DEC' in line:
        header_idx = i
        break

if header_idx is None:
    raise RuntimeError('No se encontró encabezado YEAR ... DEC en el SOI')

# Recolectar líneas con datos (4 dígitos al inicio = año) hasta fin del bloque
data_lines = []
for line in lines[header_idx + 1:]:
    s = line.strip()
    if not s: 
        # primera línea en blanco corta el bloque estandarizado
        if data_lines: break
        continue
    if not s[:4].isdigit():
        if data_lines: break
        continue
    data_lines.append(s)

print(f'SOI: {len(data_lines)} líneas de datos detectadas')

# Parsear cada línea explícitamente: YEAR + 12 valores
records = []
for line in data_lines:
    parts = line.split()
    if len(parts) < 13:
        continue
    year = int(parts[0])
    for i, mes_str in enumerate(['JAN','FEB','MAR','APR','MAY','JUN',
                                  'JUL','AUG','SEP','OCT','NOV','DEC']):
        try:
            val = float(parts[i + 1])
        except (ValueError, IndexError):
            continue
        # NOAA usa -999.9 como NoData
        if val <= -99:
            continue
        records.append({
            'date': pd.Timestamp(year=year, month=i+1, day=15),
            'soi': val,
        })

soi = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
soi = soi[(soi['date'] >= '2013-01-01') & (soi['date'] <= '2025-12-31')].copy()
print(f'SOI: {len(soi)} registros válidos, rango {soi.soi.min():.2f} a {soi.soi.max():.2f}')
print(soi.tail())

## 3. Tabla unificada de índices ENSO

Se combinan ONI y SOI en una sola tabla mensual y se exporta para uso posterior.

In [ ]:
enso = pd.merge(oni, soi, on='date', how='outer').sort_values('date').reset_index(drop=True)
enso.to_csv(OUT_TAB / 'indices_enso_mensual.csv', index=False)
print(f'Guardado: indices_enso_mensual.csv ({len(enso)} filas)')
print(enso.describe())

## 4. Visualización temporal de ONI y SOI 2013--2025

Las franjas sombreadas marcan los eventos ENSO documentados en el informe: El Niño 2015--2016 y La Niña 2020--2021. Por convención, ONI > +0.5 indica condiciones de El Niño y ONI < -0.5 condiciones de La Niña; el SOI presenta el comportamiento opuesto.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(enso.date, enso.oni, color='#D32F2F', lw=1.8, label='ONI (NOAA CPC)')
ax.plot(enso.date, enso.soi, color='#1976D2', lw=1.4, alpha=0.7, label='SOI estandarizado')
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axhspan(0.5, ax.get_ylim()[1], alpha=0.05, color='red')
ax.axhspan(ax.get_ylim()[0], -0.5, alpha=0.05, color='blue')

# Eventos ENSO destacados
ax.axvspan(pd.Timestamp('2015-06-01'), pd.Timestamp('2016-05-31'),
           alpha=0.12, color='red',  label='El Niño 2015–2016')
ax.axvspan(pd.Timestamp('2020-08-01'), pd.Timestamp('2022-03-31'),
           alpha=0.12, color='blue', label='La Niña 2020–2022')

ax.set_ylabel('Índice ENSO (estandarizado)')
ax.set_xlabel('Fecha')
ax.set_title('Índices ENSO (ONI y SOI, NOAA) sobre el periodo del proyecto, 2013–2025')
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_FIG / 'enso_serie_2013_2025.png', dpi=200, bbox_inches='tight')
plt.show()
print('Guardado: outputs/figures/enso_serie_2013_2025.png')

## 5. Correlación ONI/SOI vs anomalías NDVI por rezago temporal

Se replica la metodología del notebook `07_era5_clima.ipynb` pero sustituyendo precipitación y temperatura por los índices ENSO. Se desagregan las 8 estaciones por naturaleza espectral (manglar vs limnológica) según `tbl-clasif-estaciones` del informe, pues el promedio uniforme cancela señales opuestas entre ambos grupos.

In [ ]:
# Cargar serie NDVI z-score del proyecto
NDVI_CSV = OUT_TAB / 'serie_temporal_ndvi_definitiva.csv'
if not NDVI_CSV.exists():
    print(f'No se encontró {NDVI_CSV}'); raise SystemExit

ndvi = pd.read_csv(NDVI_CSV, parse_dates=['date'])
# Z-score por estación
ndvi['z'] = ndvi.groupby('subzona')['ndvi'].transform(
    lambda x: (x - x.mean()) / x.std())
# Mensual
ndvi['date_m'] = ndvi['date'].dt.to_period('M').dt.to_timestamp() + pd.offsets.Day(14)

# Clasificación por naturaleza
manglar      = {'Cano_Palos', 'Cano_Clarin', 'CP_Aguas_Negras', 'CP_Luna'}
limnologica  = {'Isla_Boqueron', 'Punta_Cerro', 'Punta_Chino', 'Rio_Sevilla'}
ndvi['naturaleza'] = ndvi['subzona'].apply(
    lambda s: 'manglar' if s in manglar else ('limnologica' if s in limnologica else 'otra'))

# Promedio mensual por naturaleza
z_mensual = ndvi.groupby(['date_m', 'naturaleza'])['z'].mean().reset_index()
z_mensual = z_mensual.rename(columns={'date_m': 'date'})

# Cruzar con ENSO
filas = []
for nat in ['manglar', 'limnologica']:
    serie = z_mensual[z_mensual.naturaleza == nat].merge(enso, on='date', how='inner')
    for lag in range(0, 4):
        oni_lag = serie['oni'].shift(lag)
        soi_lag = serie['soi'].shift(lag)
        rho_oni = serie['z'].corr(oni_lag)
        rho_soi = serie['z'].corr(soi_lag)
        n = (~oni_lag.isna() & ~serie['z'].isna()).sum()
        filas.append({
            'naturaleza': nat,
            'rezago_meses': lag,
            'rho_oni': round(rho_oni, 3) if pd.notna(rho_oni) else None,
            'rho_soi': round(rho_soi, 3) if pd.notna(rho_soi) else None,
            'n': int(n),
        })

df_corr = pd.DataFrame(filas)
df_corr.to_csv(OUT_TAB / 'correlacion_enso_ndvi.csv', index=False)
print(df_corr.to_string(index=False))
print(f'\nGuardado: outputs/tables/correlacion_enso_ndvi.csv')

## 6. Figura comparativa de correlaciones por rezago

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, nat in zip(axes, ['manglar', 'limnologica']):
    sub = df_corr[df_corr.naturaleza == nat]
    x = sub['rezago_meses']
    ax.bar(x - 0.18, sub['rho_oni'], width=0.36, color='#D32F2F', label='ONI')
    ax.bar(x + 0.18, sub['rho_soi'], width=0.36, color='#1976D2', label='SOI')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'Naturaleza: {nat}', fontsize=11)
    ax.set_xlabel('Rezago (meses)')
    ax.set_xticks(range(0, 4))
    ax.legend(); ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel(r'$\rho$ (Pearson) — ENSO vs NDVI z-score')
fig.suptitle('Correlación entre índices ENSO globales y anomalías NDVI por naturaleza espectral, 2013–2025',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIG / 'enso_vs_ndvi_correlacion.png', dpi=200, bbox_inches='tight')
plt.show()
print('Guardado: outputs/figures/enso_vs_ndvi_correlacion.png')

## 7. Interpretación esperada

Sobre las estaciones de manglar, la correlación negativa con ONI ---y simétricamente positiva con SOI--- a rezagos de 2 a 3 meses sería consistente con la hipótesis del informe: condiciones La Niña (ONI < −0,5, SOI > +0,5) anteceden caídas del vigor del manglar mediadas por inundación prolongada e hipoxia radicular, en tanto que sobre las estaciones limnológicas el signo se invierte porque el exceso hídrico estimula transitoriamente la señal NDVI sobre la lámina de agua.

Si las correlaciones de ENSO son más fuertes que las de ERA5-Land sobre el píxel del humedal, queda confirmado que el forzante climático opera a escala regional y no local, y que el insumo apropiado para la cadena causal son los índices de oscilación del Pacífico y no la lluvia caída sobre el AOI.